In [63]:
from pathlib import Path

import pandas as pd
import numpy as np
import requests
import re
import os
import time
import unicodedata

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 300)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

PUBLIC_RAW = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "raw"
)

RECOVERY_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "source_recovery"
)

RECOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [64]:
sites = pd.read_csv(
    REPORTS_DIR
    / "catching_sites_source_recovery.csv"
)

print(
    sites.shape
)

display(
    sites
)

(30, 4)


,site,source_recovery_status,source_path,source_url
0,Business-Website,PENDING,NaN,NaN
1,Discord-bot-website-template,PENDING,NaN,NaN
2,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,PENDING,NaN,NaN
3,academic-website-template,PENDING,NaN,NaN
4,academicpages.github.io,PENDING,NaN,NaN
5,airspace-bootstrap,PENDING,NaN,NaN
6,beginner-html-site-styled,PENDING,NaN,NaN
7,charity_responsive-template,PENDING,NaN,NaN
8,conference-website-template,PENDING,NaN,NaN
9,coreui-free-react-admin-template,PENDING,NaN,NaN


In [65]:
def normalize_repo_name(value):

    if pd.isna(value):
        return ""

    value = str(value)

    value = unicodedata.normalize(
        "NFKD",
        value
    )

    value = (
        value
        .encode(
            "ascii",
            "ignore"
        )
        .decode("ascii")
    )

    value = value.lower()

    value = re.sub(
        r"[^a-z0-9]+",
        "-",
        value
    )

    value = value.strip("-")

    return value

In [66]:
sites[
    "normalized_site"
] = sites[
    "site"
].apply(
    normalize_repo_name
)

display(
    sites[
        [
            "site",
            "normalized_site"
        ]
    ]
)

,site,normalized_site
0,Business-Website,business-website
1,Discord-bot-website-template,discord-bot-website-template
2,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,electronics-ecommerce-shop-with-admin-dashboard-nextjs-nodejs-main
3,academic-website-template,academic-website-template
4,academicpages.github.io,academicpages-github-io
5,airspace-bootstrap,airspace-bootstrap
6,beginner-html-site-styled,beginner-html-site-styled
7,charity_responsive-template,charity-responsive-template
8,conference-website-template,conference-website-template
9,coreui-free-react-admin-template,coreui-free-react-admin-template


In [67]:
GITHUB_TOKEN = os.getenv(
    "GITHUB_TOKEN"
)

headers = {
    "Accept":
        "application/vnd.github+json",

    "X-GitHub-Api-Version":
        "2022-11-28"
}

if GITHUB_TOKEN:

    headers[
        "Authorization"
    ] = (
        f"Bearer {GITHUB_TOKEN}"
    )

print(
    "GitHub token:",
    "AVAILABLE"
    if GITHUB_TOKEN
    else "NOT SET"
)

GitHub token: NOT SET


In [68]:
def search_github_repositories(
    query,
    per_page=10,
    max_retries=3
):

    url = (
        "https://api.github.com/"
        "search/repositories"
    )

    for attempt in range(
        max_retries
    ):

        response = requests.get(
            url,
            headers=headers,
            params={
                "q": query,
                "per_page": per_page
            },
            timeout=30
        )

        # Búsqueda correcta
        if response.status_code == 200:

            payload = response.json()

            return {
                "ok": True,
                "status_code": 200,
                "error": None,
                "items": payload.get(
                    "items",
                    []
                )
            }

        # Rate limit
        if response.status_code in {
            403,
            429
        }:

            remaining = response.headers.get(
                "X-RateLimit-Remaining"
            )

            reset = response.headers.get(
                "X-RateLimit-Reset"
            )

            print(
                f"Rate limit para '{query}'. "
                f"remaining={remaining}, "
                f"reset={reset}"
            )

            if reset is not None:

                wait_seconds = max(
                    int(reset)
                    - int(time.time())
                    + 2,
                    2
                )

            else:

                wait_seconds = 65

            print(
                "Esperando",
                wait_seconds,
                "segundos..."
            )

            time.sleep(
                wait_seconds
            )

            continue

        # Otro error
        return {
            "ok": False,
            "status_code":
                response.status_code,
            "error":
                response.text[:500],
            "items": []
        }

    return {
        "ok": False,
        "status_code":
            response.status_code,
        "error":
            "Maximum retries reached",
        "items": []
    }

In [69]:
test_sites = sites.head(3)

test_records = []

for _, row in test_sites.iterrows():

    result = search_github_repositories(
        row["site"]
    )

    print(
        row["site"],
        result["status_code"],
        len(result["items"])
    )

    for rank, repo in enumerate(
        result["items"],
        start=1
    ):

        test_records.append({
            "site":
                row["site"],

            "rank":
                rank,

            "repo_full_name":
                repo.get(
                    "full_name"
                ),

            "repo_name":
                repo.get(
                    "name"
                ),

            "html_url":
                repo.get(
                    "html_url"
                ),

            "description":
                repo.get(
                    "description"
                ),

            "language":
                repo.get(
                    "language"
                ),

            "stars":
                repo.get(
                    "stargazers_count"
                ),

            "archived":
                repo.get(
                    "archived"
                ),

            "fork":
                repo.get(
                    "fork"
                )
        })

test_candidates = pd.DataFrame(
    test_records
)

display(
    test_candidates
)

Business-Website 200 10
Discord-bot-website-template 200 10
Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main 200 9


,site,rank,repo_full_name,repo_name,html_url,description,language,stars,archived,fork
0,Business-Website,1,itsDaiton/business-website-template,business-website-template,https://github.com/itsDaiton/business-website-template,Modern UI/UX business website design made with React.,JavaScript,341,False,False
1,Business-Website,2,v4iv/gatsby-starter-business,gatsby-starter-business,https://github.com/v4iv/gatsby-starter-business,Gatsby Business Website Starter,JavaScript,256,False,False
2,Business-Website,3,StartBootstrap/startbootstrap-business-casual,startbootstrap-business-casual,https://github.com/StartBootstrap/startbootstrap-business-casual,A Bootstrap HTML theme for business websites - created by Start Bootstrap,Pug,361,False,False
3,Business-Website,4,treefarmstudio/odyssey-theme,odyssey-theme,https://github.com/treefarmstudio/odyssey-theme,A modern business marketing website theme/starter built with Astro 🚀,Astro,783,False,False
4,Business-Website,5,tara-pogancev/react-modern-business-website,react-modern-business-website,https://github.com/tara-pogancev/react-modern-business-website,Modern business website made using React and Bootstrap.,JavaScript,42,False,False
5,Business-Website,6,BobsProgrammingAcademy/business-website-template,business-website-template,https://github.com/BobsProgrammingAcademy/business-website-template,"A business website template for a company built using React 18 (with TypeScript 4), Material UI 5, Django 4, and Django REST Framework 3.",TypeScript,56,False,False
6,Business-Website,7,StartBootstrap/startbootstrap-small-business,startbootstrap-small-business,https://github.com/StartBootstrap/startbootstrap-small-business,A Bootstrap HTML template for creating marketing websites for small businesses - created by Start Bootstrap,JavaScript,239,False,False
7,Business-Website,8,Rebuild-Black-Business/RBB-Website,RBB-Website,https://github.com/Rebuild-Black-Business/RBB-Website,Website to help connect black-owned businesses with consumers and resources,JavaScript,121,False,False
8,Business-Website,9,vishnuraghavb/EnBizCard,EnBizCard,https://github.com/vishnuraghavb/EnBizCard,EnBizCard helps you create interactive and responsive HTML-based digital business cards that can be hosted with your website.,JavaScript,704,False,False
9,Business-Website,10,Osalumense/mld,mld,https://github.com/Osalumense/mld,A React JS Startup and small business website template designed using React and Tailwind CSS,JavaScript,185,False,False


In [70]:
candidate_records = []
search_errors = []

for i, row in sites.iterrows():

    site = row["site"]

    print(
        f"[{i + 1:02d}/{len(sites)}]",
        site
    )

    result = (
        search_github_repositories(
            site,
            per_page=10
        )
    )

    if not result["ok"]:

        search_errors.append({
            "site":
                site,

            "status_code":
                result[
                    "status_code"
                ],

            "error":
                result[
                    "error"
                ]
        })

        continue

    for rank, repo in enumerate(
        result["items"],
        start=1
    ):

        candidate_records.append({

            "site":
                site,

            "normalized_site":
                row[
                    "normalized_site"
                ],

            "rank":
                rank,

            "repo_full_name":
                repo.get(
                    "full_name"
                ),

            "repo_name":
                repo.get(
                    "name"
                ),

            "repo_normalized":
                normalize_repo_name(
                    repo.get(
                        "name"
                    )
                ),

            "html_url":
                repo.get(
                    "html_url"
                ),

            "clone_url":
                repo.get(
                    "clone_url"
                ),

            "default_branch":
                repo.get(
                    "default_branch"
                ),

            "description":
                repo.get(
                    "description"
                ),

            "language":
                repo.get(
                    "language"
                ),

            "stars":
                repo.get(
                    "stargazers_count"
                ),

            "fork":
                repo.get(
                    "fork"
                ),

            "archived":
                repo.get(
                    "archived"
                ),

            "license_key":
                (
                    repo.get(
                        "license"
                    )
                    or {}
                ).get(
                    "key"
                ),

            "license_name":
                (
                    repo.get(
                        "license"
                    )
                    or {}
                ).get(
                    "name"
                )
        })

    # Separación preventiva entre búsquedas
    if not GITHUB_TOKEN:
        time.sleep(7)

[01/30] Business-Website
[02/30] Discord-bot-website-template
[03/30] Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main
[04/30] academic-website-template
[05/30] academicpages.github.io
[06/30] airspace-bootstrap
[07/30] beginner-html-site-styled
[08/30] charity_responsive-template
[09/30] conference-website-template
[10/30] coreui-free-react-admin-template
[11/30] design-studio_one-page-template
[12/30] jekyll-inclusion
[13/30] marketing-agency_one-page-template
[14/30] material-dashboard
[15/30] personal-website-template
[16/30] portfolio_one-page-template
[17/30] react-saas-template
[18/30] simple-gallery_responsive-template
[19/30] simple-product_responsive-template
[20/30] single-page-small-business-web-application
[21/30] startbootstrap-3-col-portfolio
[22/30] startbootstrap-creative
[23/30] startbootstrap-freelancer
[24/30] startbootstrap-landing-page
[25/30] startbootstrap-one-page-wonder
[26/30] startbootstrap-resume
[27/30] startbootstrap-sb-admin-2
[28/30] s

In [71]:
candidates = pd.DataFrame(
    candidate_records
)

candidates[
    "exact_name_match"
] = (
    candidates[
        "normalized_site"
    ]
    ==
    candidates[
        "repo_normalized"
    ]
)

print(
    "Candidates:",
    candidates.shape
)

print(
    "Sitios buscados correctamente:",
    candidates[
        "site"
    ].nunique()
)

print(
    "Errores:",
    len(search_errors)
)

Candidates: (268, 17)
Sitios buscados correctamente: 30
Errores: 0


In [72]:
searched_ok = (
    len(sites)
    -
    len(search_errors)
)

print(
    "Búsquedas completadas:",
    searched_ok,
    "/",
    len(sites)
)

print(
    "Búsquedas fallidas:",
    len(search_errors)
)

Búsquedas completadas: 30 / 30
Búsquedas fallidas: 0


In [73]:
candidates[
    "exact_name_match"
].value_counts()

exact_name_match
False    195
True      73
Name: count, dtype: int64

In [74]:
exact_candidates = candidates[
    candidates[
        "exact_name_match"
    ]
].copy()

print(
    "Sitios con coincidencia exacta:",
    exact_candidates[
        "site"
    ].nunique()
)

display(
    exact_candidates[
        [
            "site",
            "repo_full_name",
            "html_url",
            "license_name",
            "archived",
            "fork"
        ]
    ]
)

Sitios con coincidencia exacta: 28


,site,repo_full_name,html_url,license_name,archived,fork
10,Discord-bot-website-template,Neovenchy/Discord-bot-website-template,https://github.com/Neovenchy/Discord-bot-website-template,MIT License,False,False
11,Discord-bot-website-template,hadi-4100/Discord-bot-website-template,https://github.com/hadi-4100/Discord-bot-website-template,MIT License,False,False
12,Discord-bot-website-template,preetsuthar17/discord-bot-website-template,https://github.com/preetsuthar17/discord-bot-website-template,NaN,False,False
13,Discord-bot-website-template,kevinjosethomas/discord-bot-website-template,https://github.com/kevinjosethomas/discord-bot-website-template,MIT License,True,False
16,Discord-bot-website-template,RedolentHalo/discord-bot-website-template,https://github.com/RedolentHalo/discord-bot-website-template,NaN,False,False
19,Discord-bot-website-template,DiscordWasMyIdea/Discord-bot-website-template,https://github.com/DiscordWasMyIdea/Discord-bot-website-template,MIT License,False,False
20,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,https://github.com/Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,MIT License,False,False
21,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,prin2603/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,https://github.com/prin2603/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,MIT License,False,False
22,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,jersonjb/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,https://github.com/jersonjb/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,MIT License,False,False
23,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,bhaskarbrsolutions/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,https://github.com/bhaskarbrsolutions/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,MIT License,False,False


In [75]:
recovery_status = (
    sites[
        [
            "site",
            "normalized_site"
        ]
    ]
    .copy()
)

recovery_status[
    "candidate_count"
] = (
    recovery_status[
        "site"
    ]
    .map(
        candidates
        .groupby("site")
        .size()
    )
    .fillna(0)
    .astype(int)
)

recovery_status[
    "exact_match_count"
] = (
    recovery_status[
        "site"
    ]
    .map(
        exact_candidates
        .groupby("site")
        .size()
    )
    .fillna(0)
    .astype(int)
)

recovery_status[
    "recovery_status"
] = np.where(
    recovery_status[
        "exact_match_count"
    ] > 0,
    "CANDIDATE",
    "UNRESOLVED"
)

recovery_status[
    "confirmed_repo"
] = None

recovery_status[
    "evidence"
] = None

In [76]:
display(
    recovery_status
)

,site,normalized_site,candidate_count,exact_match_count,recovery_status,confirmed_repo,evidence
0,Business-Website,business-website,10,0,UNRESOLVED,None,None
1,Discord-bot-website-template,discord-bot-website-template,10,6,CANDIDATE,None,None
2,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,electronics-ecommerce-shop-with-admin-dashboard-nextjs-nodejs-main,9,9,CANDIDATE,None,None
3,academic-website-template,academic-website-template,10,1,CANDIDATE,None,None
4,academicpages.github.io,academicpages-github-io,10,3,CANDIDATE,None,None
5,airspace-bootstrap,airspace-bootstrap,2,2,CANDIDATE,None,None
6,beginner-html-site-styled,beginner-html-site-styled,10,6,CANDIDATE,None,None
7,charity_responsive-template,charity-responsive-template,10,3,CANDIDATE,None,None
8,conference-website-template,conference-website-template,10,1,CANDIDATE,None,None
9,coreui-free-react-admin-template,coreui-free-react-admin-template,10,3,CANDIDATE,None,None


In [77]:
candidates.to_csv(
    REPORTS_DIR
    / "catching_source_candidates.csv",
    index=False
)

recovery_status.to_csv(
    REPORTS_DIR
    / "catching_source_recovery_status.csv",
    index=False
)

search_errors_df = pd.DataFrame(
    search_errors,
    columns=[
        "site",
        "status_code",
        "error"
    ]
)

search_errors_df.to_csv(
    REPORTS_DIR
    / "catching_source_search_errors.csv",
    index=False
)

print(
    "Errores registrados:",
    len(search_errors_df)
)

Errores registrados: 0


In [78]:
for filename in [
    "catching_source_candidates.csv",
    "catching_source_recovery_status.csv",
    "catching_source_search_errors.csv"
]:

    path = (
        REPORTS_DIR
        / filename
    )

    print(
        filename,
        "✅" if path.exists()
        else "❌"
    )

catching_source_candidates.csv ✅
catching_source_recovery_status.csv ✅
catching_source_search_errors.csv ✅


In [79]:
candidates = pd.read_csv(
    REPORTS_DIR
    / "catching_source_candidates.csv"
)

recovery = pd.read_csv(
    REPORTS_DIR
    / "catching_source_recovery_status.csv"
)

errors_path = (
    REPORTS_DIR
    / "catching_source_search_errors.csv"
)

if (
    errors_path.exists()
    and errors_path.stat().st_size > 0
):
    errors = pd.read_csv(
        errors_path
    )
else:
    errors = pd.DataFrame(
        columns=[
            "site",
            "status_code",
            "error"
        ]
    )

print(
    "Candidates:",
    candidates.shape
)

print(
    "Recovery:",
    recovery.shape
)

print(
    "Search errors:",
    errors.shape
)

Candidates: (268, 17)
Recovery: (30, 7)
Search errors: (0, 3)


In [80]:
print("=== RESUMEN RECUPERACIÓN CATCHING ===")

print(
    "\nSitios totales:",
    recovery["site"].nunique()
)

print(
    "\nEstados:"
)

print(
    recovery[
        "recovery_status"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nSitios con coincidencia exacta:",
    (
        recovery[
            "exact_match_count"
        ] > 0
    ).sum()
)

print(
    "\nSitios sin coincidencia exacta:",
    (
        recovery[
            "exact_match_count"
        ] == 0
    ).sum()
)

print(
    "\nErrores de búsqueda:",
    len(errors)
)

=== RESUMEN RECUPERACIÓN CATCHING ===

Sitios totales: 30

Estados:
recovery_status
CANDIDATE     28
UNRESOLVED     2
Name: count, dtype: int64

Sitios con coincidencia exacta: 28

Sitios sin coincidencia exacta: 2

Errores de búsqueda: 0


In [81]:
print("=== RESUMEN RECUPERACIÓN CATCHING ===")

print(
    "\nSitios totales:",
    recovery["site"].nunique()
)

print(
    "\nEstados:"
)

print(
    recovery[
        "recovery_status"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nSitios con coincidencia exacta:",
    (
        recovery[
            "exact_match_count"
        ] > 0
    ).sum()
)

print(
    "\nSitios sin coincidencia exacta:",
    (
        recovery[
            "exact_match_count"
        ] == 0
    ).sum()
)

print(
    "\nErrores de búsqueda:",
    len(errors)
)

=== RESUMEN RECUPERACIÓN CATCHING ===

Sitios totales: 30

Estados:
recovery_status
CANDIDATE     28
UNRESOLVED     2
Name: count, dtype: int64

Sitios con coincidencia exacta: 28

Sitios sin coincidencia exacta: 2

Errores de búsqueda: 0


In [82]:
exact = candidates[
    candidates[
        "exact_name_match"
    ] == True
].copy()

display(
    exact[
        [
            "site",
            "repo_full_name",
            "html_url",
            "language",
            "license_name",
            "archived",
            "fork",
            "stars"
        ]
    ]
    .sort_values(
        [
            "site",
            "stars"
        ],
        ascending=[
            True,
            False
        ]
    )
)

,site,repo_full_name,html_url,language,license_name,archived,fork,stars
10,Discord-bot-website-template,Neovenchy/Discord-bot-website-template,https://github.com/Neovenchy/Discord-bot-website-template,CSS,MIT License,False,False,453
11,Discord-bot-website-template,hadi-4100/Discord-bot-website-template,https://github.com/hadi-4100/Discord-bot-website-template,HTML,MIT License,False,False,159
13,Discord-bot-website-template,kevinjosethomas/discord-bot-website-template,https://github.com/kevinjosethomas/discord-bot-website-template,CSS,MIT License,True,False,82
12,Discord-bot-website-template,preetsuthar17/discord-bot-website-template,https://github.com/preetsuthar17/discord-bot-website-template,JavaScript,NaN,False,False,49
16,Discord-bot-website-template,RedolentHalo/discord-bot-website-template,https://github.com/RedolentHalo/discord-bot-website-template,HTML,NaN,False,False,43
19,Discord-bot-website-template,DiscordWasMyIdea/Discord-bot-website-template,https://github.com/DiscordWasMyIdea/Discord-bot-website-template,HTML,MIT License,False,False,31
24,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,shravankasar37/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,https://github.com/shravankasar37/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,TypeScript,MIT License,False,False,1
20,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,https://github.com/Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,TypeScript,MIT License,False,False,0
21,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,prin2603/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,https://github.com/prin2603/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,TypeScript,MIT License,False,False,0
22,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,jersonjb/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,https://github.com/jersonjb/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,TypeScript,MIT License,False,False,0


In [83]:
exact_summary = (
    exact
    .groupby("site")
    .agg(
        exact_repositories=(
            "repo_full_name",
            "nunique"
        ),

        repositories=(
            "repo_full_name",
            lambda x:
                " | ".join(
                    sorted(
                        set(
                            x.dropna()
                        )
                    )
                )
        )
    )
    .reset_index()
)

display(
    exact_summary
)

,site,exact_repositories,repositories
0,Discord-bot-website-template,6,DiscordWasMyIdea/Discord-bot-website-template | Neovenchy/Discord-bot-website-template | RedolentHalo/discord-bot-website-template | hadi-4100/Discord-bot-website-template | kevinjosethomas/discord-bot-website-template | preetsuthar17/discord-bot-website-template
1,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,9,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main | Uthpala773/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main | bhaskarbrsolutions/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main | emutdav/Electronics-eCommerce-Shop-With-Admin-Dashb...
2,academic-website-template,1,sbryngelson/academic-website-template
3,academicpages.github.io,3,academicpages/academicpages.github.io | arasgungore/academicpages.github.io | yangzhangalmo/academicpages.github.io
4,airspace-bootstrap,2,raffaello-templates/airspace-bootstrap | themefisher/airspace-bootstrap
5,beginner-html-site-styled,6,Salimjjuma/beginner-html-site-styled | TheBr0steam/beginner-html-site-styled | mdn/beginner-html-site-styled | sagilikhit-sys/beginner-html-site-styled | sonu7676/beginner-html-site-styled | sreemonkavungal/beginner-html-site-styled
6,charity_responsive-template,3,legndery/charity_responsive-template | mcdonnelljames44/charity-responsive-template | website-templates/charity_responsive-template
7,conference-website-template,1,mikepierce/conference-website-template
8,coreui-free-react-admin-template,3,coreui/coreui-free-react-admin-template | multicampus23/coreui-free-react-admin-template | pilotbe2man/coreui-free-react-admin-template
9,design-studio_one-page-template,2,MuhammadHaider-dev/design-studio-one-page-template | website-templates/design-studio_one-page-template


In [84]:
recovery_review = (
    recovery
    .merge(
        exact_summary,
        on="site",
        how="left"
    )
)

recovery_review[
    "exact_repositories"
] = (
    recovery_review[
        "exact_repositories"
    ]
    .fillna(0)
    .astype(int)
)

In [85]:
def preliminary_recovery_decision(row):

    n = row[
        "exact_repositories"
    ]

    if n == 0:
        return "UNRESOLVED"

    if n == 1:
        return "REVIEW_SINGLE_CANDIDATE"

    return "REVIEW_MULTIPLE_CANDIDATES"


recovery_review[
    "review_status"
] = recovery_review.apply(
    preliminary_recovery_decision,
    axis=1
)

In [86]:
print(
    recovery_review[
        "review_status"
    ].value_counts()
)

display(
    recovery_review[
        [
            "site",
            "exact_repositories",
            "repositories",
            "review_status"
        ]
    ]
)

review_status
REVIEW_MULTIPLE_CANDIDATES    17
REVIEW_SINGLE_CANDIDATE       11
UNRESOLVED                     2
Name: count, dtype: int64


,site,exact_repositories,repositories,review_status
0,Business-Website,0,NaN,UNRESOLVED
1,Discord-bot-website-template,6,DiscordWasMyIdea/Discord-bot-website-template | Neovenchy/Discord-bot-website-template | RedolentHalo/discord-bot-website-template | hadi-4100/Discord-bot-website-template | kevinjosethomas/discord-bot-website-template | preetsuthar17/discord-bot-website-template,REVIEW_MULTIPLE_CANDIDATES
2,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,9,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main | Uthpala773/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main | bhaskarbrsolutions/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main | emutdav/Electronics-eCommerce-Shop-With-Admin-Dashb...,REVIEW_MULTIPLE_CANDIDATES
3,academic-website-template,1,sbryngelson/academic-website-template,REVIEW_SINGLE_CANDIDATE
4,academicpages.github.io,3,academicpages/academicpages.github.io | arasgungore/academicpages.github.io | yangzhangalmo/academicpages.github.io,REVIEW_MULTIPLE_CANDIDATES
5,airspace-bootstrap,2,raffaello-templates/airspace-bootstrap | themefisher/airspace-bootstrap,REVIEW_MULTIPLE_CANDIDATES
6,beginner-html-site-styled,6,Salimjjuma/beginner-html-site-styled | TheBr0steam/beginner-html-site-styled | mdn/beginner-html-site-styled | sagilikhit-sys/beginner-html-site-styled | sonu7676/beginner-html-site-styled | sreemonkavungal/beginner-html-site-styled,REVIEW_MULTIPLE_CANDIDATES
7,charity_responsive-template,3,legndery/charity_responsive-template | mcdonnelljames44/charity-responsive-template | website-templates/charity_responsive-template,REVIEW_MULTIPLE_CANDIDATES
8,conference-website-template,1,mikepierce/conference-website-template,REVIEW_SINGLE_CANDIDATE
9,coreui-free-react-admin-template,3,coreui/coreui-free-react-admin-template | multicampus23/coreui-free-react-admin-template | pilotbe2man/coreui-free-react-admin-template,REVIEW_MULTIPLE_CANDIDATES


In [87]:
recovery_review.to_csv(
    REPORTS_DIR
    / "catching_source_recovery_review.csv",
    index=False
)

In [88]:
display(
    errors[
        [
            "site",
            "status_code",
            "error"
        ]
    ]
)

print(
    errors[
        "status_code"
    ].value_counts(
        dropna=False
    )
)

,site,status_code,error


Series([], Name: count, dtype: int64)
